In [1]:
from astropy.io import fits
import glob
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset

In [2]:
# Definir el Dataset
class SpectraDataset(Dataset):
    def __init__(self, flux_path, wavelength_path, redshift_path, total, num_points, scaler=None):
        """
        Parámetros:
          flux_path, wavelength_path, redshift_path: rutas a los archivos .dat
          total: número total de espectros
          num_points: número de puntos por espectro
          scaler: si se pasa, se aplica la transformación (espera un objeto StandardScaler)
        """
        self.flux = np.memmap(flux_path, dtype="float32", mode="r", shape=(total, num_points))
        self.wavelength = np.memmap(wavelength_path, dtype="float32", mode="r", shape=(total, num_points))
        self.redshift = np.memmap(redshift_path, dtype="float32", mode="r", shape=(total,))
        self.total = total
        self.num_points = num_points
        self.scaler = scaler

    def __len__(self):
        return self.total

    def __getitem__(self, idx):
        # Leer la muestra individualmente
        flux_sample = self.flux[idx, :].copy()
        wave_sample = self.wavelength[idx, :].copy()
        # Combinar en un array de forma (2, num_points)
        X = np.stack([flux_sample, wave_sample], axis=0)
        # Si se pasó un scaler, se aplica la transformación
        if self.scaler is not None:
            X_flat = X.reshape(1, -1)
            X_scaled_flat = self.scaler.transform(X_flat)
            X = X_scaled_flat.reshape(2, self.num_points)
        # Leer el redshift
        y = self.redshift[idx]
        # Convertir a tensores de PyTorch
        X_tensor = torch.tensor(X, dtype=torch.float32)
        # Se aplica unsqueeze para que el target tenga forma (1,)
        y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(0)
        return X_tensor, y_tensor

# Parámetros y rutas
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

data_dir = "data"
num_points = 5000

# Parámetros de tamaño
big_total = 595472     # Número de espectros en bigtraining
small_total = 4528     # Número de espectros en smalltraining

# Rutas de los archivos bigtraining
big_flux_path = os.path.join(data_dir, "spectra_data_bigtraining_flux.dat")
big_wavelength_path = os.path.join(data_dir, "spectra_data_bigtraining_wavelength.dat")
big_redshift_path = os.path.join(data_dir, "spectra_data_bigtraining_redshift.dat")

# Rutas de los archivos smalltraining
small_flux_path = os.path.join(data_dir, "spectra_data_smalltraining_flux.dat")
small_wavelength_path = os.path.join(data_dir, "spectra_data_smalltraining_wavelength.dat")
small_redshift_path = os.path.join(data_dir, "spectra_data_smalltraining_redshift.dat")

# # Crear los Datasets sin normalización para ajustar el scaler (comentado para no volver a hacer .fit)

# big_dataset = SpectraDataset(big_flux_path, big_wavelength_path, big_redshift_path, big_total, num_points, scaler=None)
# small_dataset = SpectraDataset(small_flux_path, small_wavelength_path, small_redshift_path, small_total, num_points, scaler=None)

# # Dividir el conjunto bigtraining en train y test (84% train, 16% test)
indices_big = np.arange(big_total)
big_train_indices, big_test_indices = train_test_split(indices_big, test_size=0.16, random_state=42)

# big_train_dataset = Subset(big_dataset, big_train_indices)
# big_test_dataset = Subset(big_dataset, big_test_indices)
#
# # Conjunto de entrenamiento final: 84% de bigtraining + todo smalltraining
# train_dataset = ConcatDataset([big_train_dataset, small_dataset])
# test_dataset = big_test_dataset  # El test proviene solo de bigtraining

# # Ajustar el scaler de forma incremental
# scaler = StandardScaler()

# # Usamos un DataLoader para iterar en batches sobre el conjunto de entrenamiento sin normalizar
# loader_for_scaler = DataLoader(train_dataset, batch_size=1024, shuffle=False, num_workers=0)
# for batch in loader_for_scaler:
#     X_batch, _ = batch  # X_batch shape: (batch_size, 2, num_points)
#     X_batch_flat = X_batch.view(X_batch.size(0), -1).numpy()  # convertir a array de NumPy
#     scaler.partial_fit(X_batch_flat)

# # Guardar el scaler para usos futuros
# scaler_path = os.path.join("extra", "scaler_modelCNN_UPD_fitted.pkl")
# with open(scaler_path, "wb") as f:
#     pickle.dump(scaler, f)
# print("Scaler ajustado y guardado en:", scaler_path)

# Cargar el scaler ya ajustado desde la ruta de guardado
scaler_path = os.path.join("extra", "scaler_modelCNN_UPD_fitted.pkl")
with open(scaler_path, 'rb') as f:
    scaler = pickle.load(f)
print("Scaler cargado desde:", scaler_path)

# Crear los Datasets con el scaler ya ajustado, asignando el scaler para que en __getitem__ se aplique la normalización
big_dataset_norm = SpectraDataset(big_flux_path, big_wavelength_path, big_redshift_path, big_total, num_points, scaler=scaler)
small_dataset_norm = SpectraDataset(small_flux_path, small_wavelength_path, small_redshift_path, small_total, num_points, scaler=scaler)

# Aplicar los mismos índices para los subconjuntos de bigtraining
big_train_dataset_norm = Subset(big_dataset_norm, big_train_indices)
big_test_dataset_norm = Subset(big_dataset_norm, big_test_indices)

# Conjunto de entrenamiento final normalizado: 84% de bigtraining + todo smalltraining
train_dataset = ConcatDataset([big_train_dataset_norm, small_dataset_norm])
test_dataset = big_test_dataset_norm

# Crear DataLoaders y pasar al dispositivo
train_loader = DataLoader(train_dataset, shuffle=True)
test_loader = DataLoader(test_dataset, shuffle=False)

print("Listo para entrenar a partir de los datos con scaler ajustado y usando memmap de forma eficiente.")

Usando dispositivo: cuda
Scaler cargado desde: extra\scaler_modelCNN_UPD_fitted.pkl
Listo para entrenar a partir de los datos con scaler ajustado y usando memmap de forma eficiente.


In [3]:
# Configurar dispositivo para GPU si está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# Número de puntos en cada espectro
num_points = 5000

# Definir el modelo CNN en PyTorch con dropout para regularización
class CNN(nn.Module):
    def __init__(self, num_points):
        super(CNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        # Tras 3 max pooling, la dimensión se reduce en un factor de 8
        conv_output_size = num_points // 8
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * conv_output_size, 128),
            nn.ReLU(),
            nn.Dropout(p=0.5),  # Regularización con Dropout
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

scaler = StandardScaler()

# Instanciar el modelo y moverlo a GPU si está disponible
modelCNN = CNN(num_points).to(device)

# Definir la función de pérdida
criterion = nn.MSELoss()

# Utilizar Adam con un weight decay para regularización L2
optimizer = optim.Adam(modelCNN.parameters(), lr=0.0001, weight_decay=1e-4)

# Scheduler adaptativo: ReduceLROnPlateau reduce la lr si la pérdida de validación no mejora
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

Usando dispositivo: cuda


In [ ]:
# Entrenar el modelo
num_epochs = 5
for epoch in range(num_epochs):
    modelCNN.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = modelCNN(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
    epoch_loss = running_loss / len(train_dataset)

    # Evaluación en el conjunto de validación
    modelCNN.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = modelCNN(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)
    val_loss /= len(test_dataset)

    # Actualizar la tasa de aprendizaje según la pérdida de validación
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, LR: {current_lr:.10f}", flush=True)

# Evaluación final en el conjunto de prueba utilizando MAE
mae_loss = nn.L1Loss()
modelCNN.eval()
test_mae = 0.0
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = modelCNN(batch_X)
        loss = mae_loss(outputs, batch_y)
        test_mae += loss.item() * batch_X.size(0)
test_mae /= len(test_dataset)
print(f"Error absoluto medio en el conjunto de prueba: {test_mae:.4f}")

# Guardar los parámetros del modelo y el scaler
checkpoint = {
    'epoch': num_epochs,
    'model_state_dict': modelCNN.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict()
}
torch.save(checkpoint, 'storage/modelCNN_UPD_100ktest.pth')

In [ ]:
# Seguir entrenando el modelo
checkpoint = torch.load('storage/modelCNN_UPD_100ktest.pth', weights_only=False)
modelCNN.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch']

# Definir el número total de epochs que deseas entrenar
num_epochs = 100

# Continuar el entrenamiento desde el epoch donde se quedó
for epoch in range(start_epoch, num_epochs + 1):
    modelCNN.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = modelCNN(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
    epoch_loss = running_loss / len(train_dataset)

    # Evaluación en el conjunto de validación
    modelCNN.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = modelCNN(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)
    val_loss /= len(test_dataset)

    # Actualizar la tasa de aprendizaje según la pérdida de validación
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    if epoch % 50 == 0:
        print(f"Epoch {epoch}/{num_epochs}, Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, LR: {current_lr:.10f}")

# Evaluación final en el conjunto de prueba utilizando MAE
mae_loss = nn.L1Loss()
modelCNN.eval()
test_mae = 0.0
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = modelCNN(batch_X)
        loss = mae_loss(outputs, batch_y)
        test_mae += loss.item() * batch_X.size(0)
test_mae /= len(test_dataset)
print(f"Error absoluto medio en el conjunto de prueba: {test_mae:.4f}")

# Guardar los parámetros del modelo y el scaler
checkpoint = {
    'epoch': num_epochs,
    'model_state_dict': modelCNN.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict()
}
torch.save(checkpoint, 'storage/modelCNN_UPD_100ktest.pth')








# Epoch 1/1200, Loss: 1.1586, Val Loss: 0.9500, LR: 0.0001000000
# Epoch 2/1200, Loss: 0.9607, Val Loss: 0.8824, LR: 0.0001000000
# Epoch 3/1200, Loss: 0.8772, Val Loss: 0.8081, LR: 0.0001000000
# Epoch 4/1200, Loss: 0.8103, Val Loss: 0.7169, LR: 0.0001000000
# Epoch 5/1200, Loss: 0.7639, Val Loss: 0.7044, LR: 0.0001000000
# Epoch 6/1200, Loss: 0.7225, Val Loss: 0.6529, LR: 0.0001000000
# Epoch 7/1200, Loss: 0.6919, Val Loss: 0.6226, LR: 0.0001000000
# Epoch 8/1200, Loss: 0.6623, Val Loss: 0.6568, LR: 0.0001000000
# Epoch 9/1200, Loss: 0.6390, Val Loss: 0.5999, LR: 0.0001000000
# Epoch 10/1200, Loss: 0.6297, Val Loss: 0.5612, LR: 0.0001000000
# Epoch 11/1200, Loss: 0.6090, Val Loss: 0.5529, LR: 0.0001000000
# Epoch 12/1200, Loss: 0.5984, Val Loss: 0.5435, LR: 0.0001000000
# Epoch 13/1200, Loss: 0.5851, Val Loss: 0.5355, LR: 0.0001000000
# Epoch 14/1200, Loss: 0.5733, Val Loss: 0.5514, LR: 0.0001000000
# Epoch 15/1200, Loss: 0.5546, Val Loss: 0.5113, LR: 0.0001000000
# Epoch 16/1200, Loss: 0.5537, Val Loss: 0.5387, LR: 0.0001000000
# Epoch 17/1200, Loss: 0.5405, Val Loss: 0.5084, LR: 0.0001000000
# Epoch 18/1200, Loss: 0.5280, Val Loss: 0.5020, LR: 0.0001000000
# Epoch 19/1200, Loss: 0.5204, Val Loss: 0.4876, LR: 0.0001000000
# Epoch 20/1200, Loss: 0.5167, Val Loss: 0.4974, LR: 0.0001000000
# Epoch 21/1200, Loss: 0.4934, Val Loss: 0.4740, LR: 0.0001000000
# Epoch 22/1200, Loss: 0.4908, Val Loss: 0.4750, LR: 0.0001000000
# Epoch 23/1200, Loss: 0.4749, Val Loss: 0.4675, LR: 0.0001000000
# Epoch 24/1200, Loss: 0.4745, Val Loss: 0.4749, LR: 0.0001000000
# Epoch 25/1200, Loss: 0.4684, Val Loss: 0.4649, LR: 0.0001000000
# Epoch 26/1200, Loss: 0.4572, Val Loss: 0.4646, LR: 0.0001000000
# Epoch 27/1200, Loss: 0.4467, Val Loss: 0.4727, LR: 0.0001000000
# Epoch 28/1200, Loss: 0.4443, Val Loss: 0.4729, LR: 0.0001000000
# Epoch 29/1200, Loss: 0.4326, Val Loss: 0.4531, LR: 0.0001000000
# Epoch 30/1200, Loss: 0.4289, Val Loss: 0.4716, LR: 0.0001000000
# Epoch 31/1200, Loss: 0.4171, Val Loss: 0.4636, LR: 0.0001000000
# Epoch 32/1200, Loss: 0.4101, Val Loss: 0.4704, LR: 0.0001000000
# Epoch 33/1200, Loss: 0.4136, Val Loss: 0.4647, LR: 0.0001000000
# Epoch 34/1200, Loss: 0.3992, Val Loss: 0.4615, LR: 0.0001000000
# Epoch 35/1200, Loss: 0.3905, Val Loss: 0.4556, LR: 0.0001000000
# Epoch 36/1200, Loss: 0.3877, Val Loss: 0.4597, LR: 0.0001000000
# Epoch 37/1200, Loss: 0.3805, Val Loss: 0.4635, LR: 0.0001000000
# Epoch 38/1200, Loss: 0.3823, Val Loss: 0.4569, LR: 0.0001000000
# Epoch 39/1200, Loss: 0.3694, Val Loss: 0.4677, LR: 0.0001000000
# Epoch 40/1200, Loss: 0.3612, Val Loss: 0.4573, LR: 0.0000500000
# Epoch 41/1200, Loss: 0.3293, Val Loss: 0.4414, LR: 0.0000500000
# Epoch 42/1200, Loss: 0.3231, Val Loss: 0.4501, LR: 0.0000500000
# Epoch 43/1200, Loss: 0.3151, Val Loss: 0.4455, LR: 0.0000500000
# Epoch 44/1200, Loss: 0.3146, Val Loss: 0.4392, LR: 0.0000500000
# Epoch 45/1200, Loss: 0.3042, Val Loss: 0.4428, LR: 0.0000500000
# Epoch 46/1200, Loss: 0.2978, Val Loss: 0.4475, LR: 0.0000500000
# Epoch 47/1200, Loss: 0.2966, Val Loss: 0.4589, LR: 0.0000500000
# Epoch 48/1200, Loss: 0.2919, Val Loss: 0.4424, LR: 0.0000500000
# Epoch 49/1200, Loss: 0.3080, Val Loss: 0.4502, LR: 0.0000500000
# Epoch 50/1200, Loss: 0.2857, Val Loss: 0.4446, LR: 0.0000500000
# Epoch 51/1200, Loss: 0.2775, Val Loss: 0.4567, LR: 0.0000500000
# Epoch 52/1200, Loss: 0.2821, Val Loss: 0.4482, LR: 0.0000500000
# Epoch 53/1200, Loss: 0.2743, Val Loss: 0.4476, LR: 0.0000500000
# Epoch 54/1200, Loss: 0.2768, Val Loss: 0.4497, LR: 0.0000500000
# Epoch 55/1200, Loss: 0.2638, Val Loss: 0.4503, LR: 0.0000250000
# Epoch 56/1200, Loss: 0.2460, Val Loss: 0.4404, LR: 0.0000250000
# Epoch 57/1200, Loss: 0.2407, Val Loss: 0.4399, LR: 0.0000250000
# Epoch 58/1200, Loss: 0.2357, Val Loss: 0.4438, LR: 0.0000250000
# Epoch 59/1200, Loss: 0.2339, Val Loss: 0.4422, LR: 0.0000250000
# Epoch 60/1200, Loss: 0.2324, Val Loss: 0.4506, LR: 0.0000250000
# Epoch 61/1200, Loss: 0.2271, Val Loss: 0.4476, LR: 0.0000250000
# Epoch 62/1200, Loss: 0.2287, Val Loss: 0.4434, LR: 0.0000250000
# Epoch 63/1200, Loss: 0.2239, Val Loss: 0.4523, LR: 0.0000250000
# Epoch 64/1200, Loss: 0.2212, Val Loss: 0.4484, LR: 0.0000250000
# Epoch 65/1200, Loss: 0.2219, Val Loss: 0.4449, LR: 0.0000250000
# Epoch 66/1200, Loss: 0.2160, Val Loss: 0.4448, LR: 0.0000125000
# Epoch 67/1200, Loss: 0.2114, Val Loss: 0.4421, LR: 0.0000125000
# Epoch 68/1200, Loss: 0.2035, Val Loss: 0.4443, LR: 0.0000125000
# Epoch 69/1200, Loss: 0.2006, Val Loss: 0.4412, LR: 0.0000125000
# Epoch 70/1200, Loss: 0.2011, Val Loss: 0.4423, LR: 0.0000125000
# Epoch 71/1200, Loss: 0.1973, Val Loss: 0.4442, LR: 0.0000125000
# Epoch 72/1200, Loss: 0.2027, Val Loss: 0.4429, LR: 0.0000125000
# Epoch 73/1200, Loss: 0.1968, Val Loss: 0.4427, LR: 0.0000125000
# Epoch 74/1200, Loss: 0.1962, Val Loss: 0.4434, LR: 0.0000125000
# Epoch 75/1200, Loss: 0.1956, Val Loss: 0.4421, LR: 0.0000125000
# Epoch 76/1200, Loss: 0.1945, Val Loss: 0.4454, LR: 0.0000125000
# Epoch 77/1200, Loss: 0.1928, Val Loss: 0.4510, LR: 0.0000062500
# Epoch 78/1200, Loss: 0.1849, Val Loss: 0.4430, LR: 0.0000062500
# Epoch 79/1200, Loss: 0.1861, Val Loss: 0.4450, LR: 0.0000062500
# Epoch 80/1200, Loss: 0.1864, Val Loss: 0.4422, LR: 0.0000062500
# Epoch 81/1200, Loss: 0.1850, Val Loss: 0.4426, LR: 0.0000062500
# Epoch 82/1200, Loss: 0.1840, Val Loss: 0.4443, LR: 0.0000062500
# Epoch 83/1200, Loss: 0.1842, Val Loss: 0.4438, LR: 0.0000062500
# Epoch 84/1200, Loss: 0.1781, Val Loss: 0.4448, LR: 0.0000062500
# Epoch 85/1200, Loss: 0.1821, Val Loss: 0.4447, LR: 0.0000062500
# Epoch 86/1200, Loss: 0.1792, Val Loss: 0.4443, LR: 0.0000062500
# Epoch 87/1200, Loss: 0.1807, Val Loss: 0.4453, LR: 0.0000062500
# Epoch 88/1200, Loss: 0.1816, Val Loss: 0.4451, LR: 0.0000031250
# Epoch 89/1200, Loss: 0.1769, Val Loss: 0.4439, LR: 0.0000031250
# Epoch 90/1200, Loss: 0.1763, Val Loss: 0.4451, LR: 0.0000031250
# Epoch 91/1200, Loss: 0.1798, Val Loss: 0.4450, LR: 0.0000031250
# Epoch 92/1200, Loss: 0.1751, Val Loss: 0.4446, LR: 0.0000031250
# Epoch 93/1200, Loss: 0.1725, Val Loss: 0.4447, LR: 0.0000031250
# Epoch 94/1200, Loss: 0.1756, Val Loss: 0.4450, LR: 0.0000031250
# Epoch 95/1200, Loss: 0.1767, Val Loss: 0.4455, LR: 0.0000031250
# Epoch 96/1200, Loss: 0.1758, Val Loss: 0.4449, LR: 0.0000031250
# Epoch 97/1200, Loss: 0.1737, Val Loss: 0.4460, LR: 0.0000031250
# Epoch 98/1200, Loss: 0.1740, Val Loss: 0.4460, LR: 0.0000031250
# Epoch 99/1200, Loss: 0.1723, Val Loss: 0.4461, LR: 0.0000015625
# Epoch 100/1200, Loss: 0.1734, Val Loss: 0.4457, LR: 0.0000015625
# Epoch 150/1200, Loss: 0.1681, Val Loss: 0.4451, LR: 0.0000000977
# Epoch 200/1200, Loss: 0.1665, Val Loss: 0.4452, LR: 0.0000000122
# Epoch 250/1200, Loss: 0.1662, Val Loss: 0.4451, LR: 0.0000000122
# Epoch 300/1200, Loss: 0.1707, Val Loss: 0.4451, LR: 0.0000000122
# Epoch 350/1200, Loss: 0.1651, Val Loss: 0.4451, LR: 0.0000000122
# Epoch 400/1200, Loss: 0.1664, Val Loss: 0.4451, LR: 0.0000000122
# Epoch 450/1200, Loss: 0.1683, Val Loss: 0.4451, LR: 0.0000000122
# Epoch 500/1200, Loss: 0.1677, Val Loss: 0.4451, LR: 0.0000000122
# Epoch 550/1200, Loss: 0.1671, Val Loss: 0.4451, LR: 0.0000000122
# Epoch 600/1200, Loss: 0.1665, Val Loss: 0.4450, LR: 0.0000000122
# Epoch 650/1200, Loss: 0.1632, Val Loss: 0.4450, LR: 0.0000000122
# Epoch 700/1200, Loss: 0.1629, Val Loss: 0.4451, LR: 0.0000000122
# Epoch 750/1200, Loss: 0.1619, Val Loss: 0.4451, LR: 0.0000000122
# Epoch 800/1200, Loss: 0.1664, Val Loss: 0.4451, LR: 0.0000000122
# Epoch 850/1200, Loss: 0.1674, Val Loss: 0.4451, LR: 0.0000000122
# Epoch 900/1200, Loss: 0.1636, Val Loss: 0.4450, LR: 0.0000000122
# Epoch 950/1200, Loss: 0.1647, Val Loss: 0.4450, LR: 0.0000000122
# Epoch 1000/1200, Loss: 0.1660, Val Loss: 0.4451, LR: 0.0000000122
# Epoch 1050/1200, Loss: 0.1666, Val Loss: 0.4451, LR: 0.0000000122
# Epoch 1100/1200, Loss: 0.1660, Val Loss: 0.4450, LR: 0.0000000122
# Epoch 1150/1200, Loss: 0.1637, Val Loss: 0.4450, LR: 0.0000000122
# Epoch 1200/1200, Loss: 0.1649, Val Loss: 0.4450, LR: 0.0000000122

In [ ]:
# Obtener la lista inicial de archivos FITS
folder_path = r'spectrums'
files = [f for f in os.listdir(folder_path) if f.endswith('.fits')]
files = random.sample(files, len(files))
file_path = os.path.join(folder_path, files[0])

checkpoint = torch.load('storage/modelCNN_UPD_100ktest.pth', weights_only=False)
modelCNN.load_state_dict(checkpoint['model_state_dict'])
with open('extra/scaler_modelCNN_UPD_100ktest.pkl', 'rb') as f:
    scaler = pickle.load(f)


with fits.open(file_path) as hdul:
    test_flux = hdul[1].data["flux"]
    test_loglam = hdul[1].data["loglam"]
    test_redshift = hdul[2].data["Z"][0]  # Asumiendo que Z es un array y queremos el primer valor

test_wavelength = 10 ** test_loglam

def expand_points(wavelength, flux, target_count=5000):
    # Convertir a listas para facilitar las inserciones
    wl = list(wavelength)
    fl = list(flux)
    
    # Calcular las diferencias absolutas entre puntos consecutivos
    diffs = [abs(fl[i+1] - fl[i]) for i in range(len(fl)-1)]
    # Obtener los índices ordenados de mayor a menor diferencia
    sorted_indices = sorted(range(len(diffs)), key=lambda i: diffs[i], reverse=True)
    
    # Insertar nuevos puntos utilizando los índices ordenados
    while len(wl) < target_count:
        # Se recorre la lista de índices en orden descendente para evitar problemas con el reordenamiento
        for idx in sorted_indices:
            if len(wl) >= target_count:
                break
            # Calcular la interpolación lineal entre el punto idx y el siguiente
            new_wl = (wl[idx] + wl[idx+1]) / 2
            new_fl = (fl[idx] + fl[idx+1]) / 2
            # Insertar el nuevo punto en la posición correspondiente
            wl.insert(idx+1, new_wl)
            fl.insert(idx+1, new_fl)
    
    return np.array(wl), np.array(fl)

test_wavelength, test_flux = expand_points(test_wavelength, test_flux, target_count=num_points)

# Preprocesamiento para el modelo
# Canal 0: flux; Canal 1: wavelength
input_data = np.stack([test_flux, test_wavelength], axis=0)  # (2, 5000) 2 canales de tamaño 5000
input_data = input_data.reshape(1, 2, num_points)            # (1, 2, 5000) 1 muestra, 2 canales, 5000 puntos/canal

# Normalización
nsamples, nchannels, npoints = input_data.shape # Guardar dimensionalidad inicial
input_flat = input_data.reshape(nsamples, -1) # Aplanar/concatenar (1, 10000)
input_scaled = scaler.transform(input_flat) # Normalizar
input_scaled = input_scaled.reshape(nsamples, nchannels, npoints) # Recuperar dimensionalidad incial

# Convertir a tensor
input_tensor = torch.tensor(input_scaled, dtype=torch.float32)
input_tensor = input_tensor.to(device)

# Evaluar el modelo
with torch.no_grad():
    predicted_redshift = modelCNN(input_tensor)

print("Redshift real:", test_redshift)
print("Redshift predicho:", predicted_redshift.item())

plt.figure(figsize=(12, 6))
plt.plot(test_wavelength, test_flux, label="Test Espectro")
plt.xlabel("Longitud de onda (Ångstrom)")
plt.ylabel("Flujo (10^-17 erg/s/cm²/Å)")
plt.title("Espectro vs. Flujo (Test)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()